# 4-2 ReLU 심화

강의 원문 대신 직접 작성하고 실행한 코드와 학습 메모를 정리했습니다.


In [4]:
import torch
import torch.nn as nn

X = torch.tensor([[1., 1.], [2., 1.]])
W = torch.tensor([[-1., -1.], [1., 0.]])
b = torch.tensor([-1., -0.5])

pre = X @ W.T + b
act = torch.relu(pre)
act_ratio = (act > 0).float().mean(dim=0)

print(pre)
print(act)
print(act_ratio)
print("inspect_neuron:", (torch.argmin(act_ratio)))

tensor([[-3.0000,  0.5000],
        [-4.0000,  1.5000]])
tensor([[0.0000, 0.5000],
        [0.0000, 1.5000]])
tensor([0., 1.])
inspect_neuron: tensor(0)


In [5]:

class AuditedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(2, 2)

    def forward(self, x):
        hidden = self.relu(self.fc1(x))  # 값은 자르지만 (B,2) shape는 유지합니다.
        logits = self.fc2(hidden)         # 출력층은 음수도 허용하는 raw 점수입니다.
        return logits, hidden

torch.manual_seed(4)
model = AuditedMLP()
with torch.no_grad():
    # 고정 weight로 hidden 양수 2개/전체 4개를 만들며 대표 출력을 재현합니다.
    model.fc1.weight.copy_(torch.eye(2))
    model.fc1.bias.copy_(torch.tensor([-0.75, 0.0]))
logits, hidden = model(torch.tensor([[1., -1.], [0.5, 2.]]))
assert hidden.shape == logits.shape == (2, 2)
print("shapes:", tuple(hidden.shape), tuple(logits.shape))
print("active_ratio:", round(float((hidden > 0).float().mean()), 2))

shapes: (2, 2) (2, 2)
active_ratio: 0.5


In [8]:
# 검증 가능 정답 코드
runs = {
    "A": {"ratio": 0.00, "acc": 0.82, "zero_batches": 5},
    "B": {"ratio": 0.48, "acc": 0.84, "zero_batches": 0},
    "C": {"ratio": 0.92, "acc": 0.86, "zero_batches": 0},
}
ready, investigate, blocked = [], [], []
# 반복된 완전 비활성은 차단하되, 극단적 활성 비율은 곧바로 탈락시키지 않고 조사 상태로 분리합니다.
for name, run in runs.items():
    if run["acc"] < 0.80 or run["zero_batches"] >= 3:
        blocked.append(name)
    elif not 0.25 <= run["ratio"] <= 0.75:
        investigate.append(name)
    else:
        ready.append(name)
# 배포 후보 선택은 조사·차단 실행을 제외한 ready 집합 안에서만 수행합니다.
selected = max(ready, key=lambda name: runs[name]["acc"]) if ready else "보류"
print("ready/investigate/blocked:", ready, investigate, blocked)
print("selected:", selected)

ready/investigate/blocked: ['B'] ['C'] ['A']
selected: B
